# Simple: Numeric Field Analysis with PAMOLA.CORE

**Goal**: Learn numeric field analysis basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze numeric fields using execute()
- Compare statistical distributions
- Understand outliers and normality

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 01_numeric_analyzer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.numeric import NumericOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data if file missing
- Review dataset structure before proceeding

**What you'll see:**
- Dataset summary (100 records, 7 columns)
- First 5 records preview
- Column statistics with types and ranges
- Numeric fields: salary, age, experience_years, performance_score

**Note:** Sample includes intentional outliers and nulls for analysis demonstration

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with numeric fields
    np.random.seed(42)
    n_samples = 100
    
    sample_data = pd.DataFrame({
        'record_id': range(1, n_samples + 1),
        'salary': np.random.normal(75000, 20000, n_samples).round(2),
        'age': np.random.randint(22, 65, n_samples),
        'experience_years': np.random.randint(0, 40, n_samples),
        'performance_score': np.random.uniform(1.0, 5.0, n_samples).round(2),
        'project_count': np.random.poisson(5, n_samples),
        'satisfaction_rating': np.random.choice([1, 2, 3, 4, 5], n_samples, p=[0.05, 0.10, 0.20, 0.35, 0.30])
    })
    
    # Add some outliers
    sample_data.loc[5, 'salary'] = 250000  # High outlier
    sample_data.loc[15, 'salary'] = 15000   # Low outlier
    sample_data.loc[25, 'age'] = 70         # Age outlier
    
    # Add some null values
    sample_data.loc[np.random.choice(n_samples, 5, replace=False), 'performance_score'] = np.nan
    
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name** in `create_config_kwargs()`
   - Default: `"current_salary_cad"`
   - Change to YOUR dataset's numeric column name
2. Run to validate field and setup environment

**What you'll see:**
- Field validation (type, range, mean, median)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and progress tracker ready (✓)

**Note:** Field must exist in dataset and be numeric type (int or float)

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "current_salary_cad"  # Field to analyze - CUSTOMIZE THIS!
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to process: '{field_name}'")
print(f"  Data type: {df[field_name].dtype}")
print(f"  Non-null count: {df[field_name].notna().sum()}")
print(f"  Range: {df[field_name].min():.2f} to {df[field_name].max():.2f}")
print(f"  Mean: {df[field_name].mean():.2f}")
print(f"  Median: {df[field_name].median():.2f}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="numeric_analysis",
    description="Numeric field analysis of salary",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration parameters below
- Run to execute numeric field analysis
- Monitor progress bar (6 steps: load → validate → analyze → stats → outliers → complete)

**Key parameters:**
- `field_name`: Field to analyze (numeric required)
- `bins=10`: Number of histogram bins
- `detect_outliers=True`: IQR-based outlier detection
- `test_normality=True`: Shapiro-Wilk, Anderson-Darling tests
- `generate_visualization=True`: Creates histogram, boxplot, Q-Q plot

**What you'll see:**
- Configuration summary with all parameters
- Progress bar: validation → statistics → outliers → normality → save → complete
- Completion status: "completed" (verify no errors)
- Artifact count (4-6 files expected)

**Note:** Analysis is non-destructive, original data unchanged

In [ ]:
# Create operation with production-style configuration
operation = NumericOperation(
    # Core parameters
    field_name=field_name,
    bins=10,
    detect_outliers=True,
    test_normality=True,
    near_zero_threshold=1e-10,
    
    # Output configuration
    output_format='json',
    profile_type='numeric',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    chunk_size=10000,
    use_dask=False,
    npartitions=None,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Bins:             {operation.bins}")
print(f"  Detect outliers:  {operation.detect_outliers}")
print(f"  Test normality:   {operation.test_normality}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing numeric analysis...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load statistical analysis from task_dir
- Review comprehensive metrics output
- Verify key statistics match expectations

**What you'll see:**
- Basic statistics (count, mean, median, std dev)
- Distribution shape (skewness, kurtosis)
- Outlier detection results (IQR method)
- Normality test results (Shapiro-Wilk, Anderson-Darling)
- Percentile breakdown (P1, P5, P25, P50, P75, P95, P99)

**Note:** "is_normal: True" means data follows normal distribution (p-value > 0.05)

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.json'))
if output_files:
    # Find the stats file
    stats_file = [f for f in output_files if 'stats' in f.name]
    if stats_file:
        with open(stats_file[0], 'r') as f:
            stats_data = json.load(f)
        
        print("\n" + "=" * 80)
        print("📊 STATISTICAL ANALYSIS")
        print("=" * 80)
        
        # Display basic statistics
        print(f"\n📈 Basic Statistics:")
        print(f"  Total rows:       {stats_data.get('total_rows', 0):,}")
        print(f"  Valid values:     {stats_data.get('valid_count', 0):,}")
        print(f"  Null values:      {stats_data.get('null_count', 0):,} ({stats_data.get('null_percentage', 0):.2f}%)")
        
        stats = stats_data.get('stats', {})
        if stats:
            print(f"\n📊 Descriptive Statistics:")
            print(f"  Min:              {stats.get('min', 0):,.2f}")
            print(f"  Max:              {stats.get('max', 0):,.2f}")
            print(f"  Mean:             {stats.get('mean', 0):,.2f}")
            print(f"  Median:           {stats.get('median', 0):,.2f}")
            print(f"  Std Dev:          {stats.get('std', 0):,.2f}")
            print(f"  Variance:         {stats.get('variance', 0):,.2f}")
            print(f"  Skewness:         {stats.get('skewness', 0):.4f}")
            print(f"  Kurtosis:         {stats.get('kurtosis', 0):.4f}")
            
            # Value distribution
            print(f"\n🔢 Value Distribution:")
            print(f"  Zero values:      {stats.get('zero_count', 0):,} ({stats.get('zero_percentage', 0):.2f}%)")
            print(f"  Positive values:  {stats.get('positive_count', 0):,} ({stats.get('positive_percentage', 0):.2f}%)")
            print(f"  Negative values:  {stats.get('negative_count', 0):,} ({stats.get('negative_percentage', 0):.2f}%)")
            
            # Outlier analysis
            if 'outliers' in stats:
                outliers = stats['outliers']
                print(f"\n⚠️  Outlier Detection (IQR method):")
                print(f"  IQR:              {outliers.get('iqr', 0):,.2f}")
                print(f"  Lower bound:      {outliers.get('lower_bound', 0):,.2f}")
                print(f"  Upper bound:      {outliers.get('upper_bound', 0):,.2f}")
                print(f"  Outliers found:   {outliers.get('count', 0):,} ({outliers.get('percentage', 0):.2f}%)")
                if 'sample' in outliers and outliers['sample']:
                    print(f"  Sample outliers:  {[f'{x:.2f}' for x in outliers['sample'][:5]]}")
            
            # Normality testing
            if 'normality' in stats:
                normality = stats['normality']
                print(f"\n📊 Normality Testing:")
                is_normal = normality.get('is_normal', False)
                print(f"  Is Normal:        {'✓ Yes' if is_normal else '✗ No'}")
                
                if 'shapiro' in normality:
                    shapiro = normality['shapiro']
                    print(f"  Shapiro-Wilk:     statistic={shapiro.get('statistic', 0):.4f}, p-value={shapiro.get('p_value', 0):.4f}")
                
                if 'anderson' in normality:
                    anderson = normality['anderson']
                    print(f"  Anderson-Darling: statistic={anderson.get('statistic', 0):.4f}")
            
            # Percentiles
            if 'percentiles' in stats:
                percentiles = stats['percentiles']
                print(f"\n📊 Key Percentiles:")
                for p in ['p1', 'p5', 'p25', 'p50', 'p75', 'p95', 'p99']:
                    if p in percentiles:
                        print(f"  {p.upper():6s}:          {percentiles[p]:,.2f}")
        
        print("\n" + "=" * 80)
        print("✨ SUMMARY")
        print("=" * 80)
        print(f"  Dataset size:     {stats_data.get('total_rows', 0):,} records")
        print(f"  Valid data:       {stats_data.get('valid_count', 0):,} values")
        if stats.get('outliers'):
            print(f"  Outliers:         {stats['outliers'].get('count', 0):,} detected")
        if stats.get('normality'):
            is_normal = stats['normality'].get('is_normal', False)
            print(f"  Distribution:     {'Normal' if is_normal else 'Non-normal'}")
        
        print(f"\n💡 Statistical analysis complete!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-2 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Statistical analysis JSON
├── metrics/          # Performance metrics JSON
├── visualizations/   # Histogram, boxplot, Q-Q plot (PNG)
└── config/           # Operation configuration
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance and statistical metrics
2. **Output Data**: Statistical analysis results
3. **Visualizations**: Histogram, boxplot, and Q-Q plot

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. OUTPUT DATA (NEWEST - Stats JSON)
print("\n1️⃣ STATISTICAL ANALYSIS (JSON):")
print("-" * 80)

output_dir = task_dir / "output"
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob("*.json"))

    if not output_files:
        print("⚠️  No output files found")
    else:
        # Newest first
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]

        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")

        try:
            with open(latest_output_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            # BASIC SUMMARY
            print("📊 BASIC SUMMARY:")
            print(f"   Total Rows     : {data.get('total_rows', 0):,}")
            print(f"   Valid Count    : {data.get('valid_count', 0):,}")
            print(f"   Null Count     : {data.get('null_count', 0):,}")
            print(f"   Null Percentage: {data.get('null_percentage', 0):.2f}%")

            stats = data.get("stats", {})
            if not stats:
                print("\n⚠️  No statistical details found")
            else:
                # CORE STATISTICS
                print("\n📈 CORE STATISTICS:")
                for key in ["min", "max", "mean", "median", "std", "variance", "sum"]:
                    if key in stats:
                        print(f"   {key.capitalize():<10}: {stats[key]:,.2f}")

                # DISTRIBUTION SHAPE
                print("\n📐 DISTRIBUTION SHAPE:")
                print(f"   Skewness : {stats.get('skewness', 0):.4f}")
                print(f"   Kurtosis : {stats.get('kurtosis', 0):.4f}")

                # VALUE SIGN ANALYSIS
                print("\n➕➖ VALUE SIGN ANALYSIS:")
                print(f"   Positive : {stats.get('positive_count', 0):,} ({stats.get('positive_percentage', 0):.2f}%)")
                print(f"   Zero     : {stats.get('zero_count', 0):,} ({stats.get('zero_percentage', 0):.2f}%)")
                print(f"   Negative : {stats.get('negative_count', 0):,} ({stats.get('negative_percentage', 0):.2f}%)")

                # PERCENTILES
                percentiles = stats.get("percentiles")
                if percentiles:
                    print("\n📊 PERCENTILES:")
                    for k, v in percentiles.items():
                        print(f"   {k:>6}: {v:,.2f}")

                # HISTOGRAM
                histogram = stats.get("histogram")
                if histogram and histogram.get("bins"):
                    print(f"\n📦 HISTOGRAM ({len(histogram['bins'])} bins):")
                    for b, c in zip(histogram["bins"][:5], histogram["counts"][:5]):
                        print(f"   {b:<25} : {c}")
                    if len(histogram["bins"]) > 5:
                        print("   ...")

                # OUTLIERS
                outliers = stats.get("outliers")
                if outliers:
                    print("\n🚨 OUTLIERS (IQR METHOD):")
                    print(f"   Count      : {outliers.get('count', 0)}")
                    print(f"   Percentage : {outliers.get('percentage', 0):.2f}%")
                    print(f"   Bounds     : [{outliers.get('lower_bound', 0):,.2f} , {outliers.get('upper_bound', 0):,.2f}]")

                    samples = outliers.get("sample", [])
                    if samples:
                        print(f"   Sample     : {', '.join(f'{v:,.0f}' for v in samples[:10])}")

                # NORMALITY TESTS
                normality = stats.get("normality")
                if normality:
                    print("\n🧪 NORMALITY TESTS:")
                    print(f"   Overall     : {'Normal' if normality.get('is_normal') else 'Non-normal'}")
                    print(f"   Tests Passed: {normality.get('normal_test_passed', 0)} / {normality.get('normal_test_count', 0)}")

                    for test in ["shapiro", "anderson", "ks"]:
                        if test in normality:
                            t = normality[test]
                            print(
                                f"   {test.capitalize():<10}: "
                                f"stat={t.get('statistic', 0):.4f}, "
                                f"p={t.get('p_value', float('nan')):.2e}, "
                                f"normal={t.get('normal', False)}"
                            )
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 2. VISUALIZATIONS (NEWEST BATCH)
print("\n\n2️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                if 'distribution' in viz_file.name:
                    viz_name = 'Distribution Histogram'
                elif 'boxplot' in viz_file.name:
                    viz_name = 'Boxplot'
                elif 'qqplot' in viz_file.name:
                    viz_name = 'Q-Q Plot (Normality Test)'
                else:
                    viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure numeric analysis with full parameters  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Numeric analysis reveals distribution patterns
- Outlier detection identifies anomalies
- Normality tests inform statistical methods
- Descriptive statistics summarize data
- All artifacts saved in structured directories

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_numeric_analyzer_advanced.ipynb`** includes:
- Large dataset processing with Dask
- Parallel processing with Joblib
- Custom normality test methods
- Advanced outlier detection
- Performance optimization
- Correlation analysis

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)